# Week 8 — Multilingual Campaign Generator
**Tools:** Gemini 1.5 Flash API
**Datasets:** sloganlist.csv (Company, Slogan) → translation targets | startups.csv (name, city, tagline) → target regions

In [ ]:
!pip install google-generativeai pandas -q
import google.generativeai as genai,pandas as pd,json,re
from google.colab import userdata,drive
drive.mount('/content/drive')
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gm=genai.GenerativeModel('gemini-1.5-flash')
print('Gemini ready ✓')

In [ ]:
BASE='/content/drive/MyDrive/BrandSphere/'
sl=pd.read_csv(BASE+'sloganlist.csv'); sl.columns=['Company','Slogan']
st=pd.read_csv(BASE+'startups.csv').dropna(subset=['tagline'])
print('Slogans:',len(sl),'| Startups:',len(st))
# Load week 4 taglines if available
try:
    with open('week4_output.json') as f: taglines=json.load(f)['taglines']
except: taglines=['Powering tomorrow, today.','Where intelligence meets execution.','Your workflow, amplified.']

In [ ]:
TARGET_LANGS=['Hindi','Spanish','French','Mandarin','Arabic','Portuguese','German']
def translate_content(text,langs=TARGET_LANGS,company='NovaTech',tone='Tech-Forward'):
    p=f"""Translate this brand tagline into: {', '.join(langs)}\n\nOriginal: \"{text}\"\nBrand: {company} | Tone: {tone}\n\nRules: Preserve tone | Sound natural | Adapt culturally | Stay punchy\nReturn ONLY valid JSON: {{{', '.join(f'"{l}": "..."' for l in langs)}}}"""
    raw=gm.generate_content(p).text.strip(); m=re.search(r'\{.*?\}',raw,re.DOTALL)
    return json.loads(m.group()) if m else {'error':raw}
all_tr={}
for tl in taglines:
    tr=translate_content(tl); all_tr[tl]=tr
    print(f'\nOriginal: {tl}')
    for l,t in tr.items(): print(f'  {l:12s}: {t}')

In [ ]:
# Multilingual campaign posts
def gen_multilingual_post(tagline,lang,company='NovaTech',industry='Technology'):
    p=f"""Create a {lang}-language Instagram caption for {company} ({industry}).\nTagline: \"{tagline}\"\nUnder 120 chars. Include 3 hashtags in {lang}. Sound natural and culturally appropriate.\nReturn ONLY JSON: {{\"caption\":\"...\",\"hashtags\":[\"...\",\"...\",\"...\"]}}"""
    raw=gm.generate_content(p).text.strip(); m=re.search(r'\{.*?\}',raw,re.DOTALL)
    return json.loads(m.group()) if m else {}
# Test with Hindi and Spanish
for lang in ['Hindi','Spanish']:
    post=gen_multilingual_post(taglines[0],lang)
    print(f'\n{lang} Post:'); [print(f'  {k}: {v}') for k,v in post.items()]

In [ ]:
with open('week8_translations.json','w',encoding='utf-8') as f: json.dump(all_tr,f,ensure_ascii=False,indent=2)
print('Saved week8_translations.json ✓')
# Also CSV for easy reading
rows=[]
for src,trs in all_tr.items():
    for lang,text in trs.items(): rows.append({'Source_English':src,'Language':lang,'Translation':text})
pd.DataFrame(rows).to_csv('week8_translations.csv',index=False)
print('Saved week8_translations.csv ✓')

## ✅ Week 8 Deliverables
- [ ] sloganlist.csv (Company, Slogan) used for translation targets
- [ ] startups.csv (name, city, tagline, description) used for regional audience mapping
- [ ] 7-language translation: Hindi, Spanish, French, Mandarin, Arabic, Portuguese, German
- [ ] Multilingual Instagram post generator per language
- [ ] Saved: week8_translations.json, week8_translations.csv